Generate a chart from a preprocessed `.pt` song

Runs top to bottom: setup, load model, define the generator, then set song settings and run.

In [16]:
from pathlib import Path

from tracers.project import setup_project
from tracers.pipelines.trainer_factory import build_trainer

PROJECT_ROOT = Path.cwd().parent
setup = setup_project(PROJECT_ROOT)

MODEL_KEY = "tracer_np"

experiment_config = setup["experiment_config"]
hyperparams = setup["hyperparams"]
segment_frames = setup["segment_frames"]
device = setup["device"]

In [17]:
from tracers.pipelines.generate import run_custom_generation

trainer, model = build_trainer(
    model_key=MODEL_KEY,
    hyperparams=hyperparams,
    experiment_config=experiment_config,
    device=device,
    checkpoint_path=str(PROJECT_ROOT / "notebooks" / "checkpoints" / "tracer_np" / "best.pt"),
    train_loader=None,
    valid_loader=None,
)

Loaded c:\Users\joshua\vsjycode\csc173\transformers\TRACERS\notebooks\checkpoints\tracer_np\best.pt (epoch 76, best val loss 0.1494)


In [18]:
from tracers.export.chart import export_generated_chart

# ---------- settings ----------
AUDIO_PT_NAME = "nichijou"
AUDIO_PT = PROJECT_ROOT / f"{AUDIO_PT_NAME}.pt"
DIFFICULTY = 3
BPM = 157
OFFSET_MS = 0
LANE_MAP = (0, 2, 4, 6)

TEMPERATURE = 0.35
TOP_K = None
TOP_P = None
SEED = 0
SAVE_DEBUG_CSV = True

# ---------- generate ----------
generated, debug_segments = run_custom_generation(
    trainer,
    AUDIO_PT,
    difficulty=DIFFICULTY,
    segment_frames=segment_frames,
    context_frames=500,
    seed_action=0,
    seed=SEED,
    temperature=TEMPERATURE,
    top_k=TOP_K,
    top_p=TOP_P,
    collect_debug=SAVE_DEBUG_CSV,
)

# ---------- export ----------
output_dir = Path(f"charts/{AUDIO_PT_NAME}")
output_dir.mkdir(parents=True, exist_ok=True)
export_generated_chart(
    generated,
    output_dir / "generated_chart.json",
    bpm=BPM,
    start_frame=0,
    offset_ms=OFFSET_MS,
    lane_map=LANE_MAP,
)
print("Saved generated_chart.json")

if SAVE_DEBUG_CSV:
    from tracers.generate.converter import export_generation_debug
    export_generation_debug(debug_segments, output_path=str(output_dir / "generation_debug.csv"), top_k=5)

Prediction frames: 24808 (248.1 s)
Window size: 2000 | Context: 500 | Stride: 1500
Window      1-  2000 | Context:    0 | New: 2000 | Active:   42
Window   1501-  3500 | Context:  500 | New: 1500 | Active:   45
Window   3001-  5000 | Context:  500 | New: 1500 | Active:   41
Window   4501-  6500 | Context:  500 | New: 1500 | Active:   44
Window   6001-  8000 | Context:  500 | New: 1500 | Active:   53
Window   7501-  9500 | Context:  500 | New: 1500 | Active:   47
Window   9001- 11000 | Context:  500 | New: 1500 | Active:   32
Window  10501- 12500 | Context:  500 | New: 1500 | Active:   38
Window  12001- 14000 | Context:  500 | New: 1500 | Active:   40
Window  13501- 15500 | Context:  500 | New: 1500 | Active:   46
Window  15001- 17000 | Context:  500 | New: 1500 | Active:   42
Window  16501- 18500 | Context:  500 | New: 1500 | Active:   48
Window  18001- 20000 | Context:  500 | New: 1500 | Active:   32
Window  19501- 21500 | Context:  500 | New: 1500 | Active:   43
Window  21001- 23000 